# Test Predict For Workout

## Import libraries

In [5]:
import pandas as pd
import numpy as np
import pickle
from IPython.display import display

## load Model

In [6]:
# LOAD THE MODEL
path = '../../models/model_workout.pickle'

with open(path, 'rb') as f:
    model_data = pickle.load(f)
    
knn = model_data['knn_model']           # The Brain (Trained with Supernova Weights)
scaler = model_data['scaler']           # The Scaler (MinMax)
weights = model_data['weights']         # The Rules (Weights dictionary)
db_profiles = model_data['profiles_db'] # The User Database
db_schedule = model_data['schedule_db'] # The Workout Logs
encoders = model_data['encoders']       # The Encoders
feature_order = model_data['features']  # The Feature Map

print("✅ Workout Model Loaded & Ready.")

✅ Workout Model Loaded & Ready.


## Prediciton (Helper) Function

In [7]:
# PREDICTION FUNCTION

def predict_workout_plan(user_input):
    """
    Generates a fully personalized weekly workout schedule using a Weighted 
    K-Nearest Neighbors (KNN) algorithm.

    This function identifies the most similar user in the database based on physical 
    attributes while STRICTLY enforcing logistical constraints (Goal, Frequency, 
    Environment) using a custom weighting mechanism.

    ---------------------------------------------------------------------------
    Parameters (Input):
    1. user_input (dict):
        A dictionary containing the user's profile and preferences. 
        Required Keys:
        - 'Age' (int)        : User's age.
        - 'Gender' (str)     : 'Male' or 'Female'.
        - 'Weight' (float)   : User's weight in kg.
        - 'Goal' (str)       : e.g., 'Muscle Gain', 'Weight Loss'.
        - 'Frequency' (int)  : Desired workout days per week (1-7).
        - 'Level' (str)      : 'Beginner', 'Intermediate', or 'Advanced'.
        - 'Environment' (str): 'Home' or 'Gym'.
        - Cardio Flags (int) : 1 (Can do) or 0 (Cannot do) for:
          'Badminton', 'Football', 'Basketball', 'Volleyball', 'Swim'.

    ---------------------------------------------------------------------------
    Logic & Mechanism:
    
    A. INPUT ENCODING:
    Converts categorical text inputs (e.g., 'Male', 'Home') into numerical 
    format using pre-trained LabelEncoders.

    B. FEATURE MAPPING & SCALING:
    Aligns user input with the model's feature structure and normalizes values 
    (0-1 range) to ensure mathematical consistency.

    C. SUPERNOVA WEIGHTING (The Core Logic):
    Applies massive weight multipliers (x100) to critical constraints:
    - Goal, Frequency, Environment, and Cardio Capabilities.
    
    Why? This forces the KNN algorithm to treat a mismatch in these fields 
    as a "catastrophic error." 
    Example: If a user requests a 3-Day Home plan, the model is mathematically 
    forbidden from suggesting a 5-Day Gym plan, even if the body types match perfectly.

    D. NEAREST NEIGHBOR SEARCH:
    Locates the single best matching user profile (n_neighbors=1) in the 
    weighted multi-dimensional space.

    ---------------------------------------------------------------------------
    Returns:
    - pandas.DataFrame: A schedule containing the matched user's workout routine 
      (Day, Exercise Name, Sets, Reps, etc.).
    - Prints a debug report to the console showing the matched User ID and 
      verification of the constraint matching.
    """
    
    # --- STEP 1: ENCODE USER INPUT ---
    # Convert text inputs (e.g., 'Male', 'Home') into numbers (0, 1, 2...)
    try:
        goal_enc = encoders['goal'].transform([user_input['Goal']])[0]
        level_enc = encoders['level'].transform([user_input['Level']])[0]
        gender_enc = encoders['gender'].transform([user_input['Gender']])[0]
        env_enc = encoders['environment'].transform([user_input['Environment']])[0]
    except ValueError as e:
        print(f"❌ Error: Invalid Category found. {e}")
        return None

    # --- STEP 2: PREPARE FEATURE VECTOR ---
    # Map inputs to the exact column order the model expects
    input_data = {
        'Goal_Encoded': goal_enc,
        'Workout_Frequency_x': user_input['Frequency'],
        'level_Encoded': level_enc,
        'Gender_Encoded': gender_enc,
        'Age_x': user_input['Age'],
        'Initial_Weight_kg_x': user_input['Weight'],
        'environment_Encoded': env_enc,
        
        # Cardio Constraints (Binary Flags 1/0)
        'Badminton': user_input['Badminton'],
        'Football': user_input['Football'],
        'Basketball': user_input['Basketball'],
        'Volleyball': user_input['Volleyball'],
        'Swim': user_input['Swim']
    }
    
    # Convert to DataFrame
    input_df = pd.DataFrame([input_data])[feature_order]
    
    # --- STEP 3: SCALE & WEIGHT ---
    # A. Scale features to 0-1 range
    input_scaled = scaler.transform(input_df)
    
    # B. Apply "Supernova Weights" (Multiply by 100 where needed)
    # This enforces the strict rules (Frequency, Environment, Sports)
    input_weighted = pd.DataFrame(input_scaled, columns=feature_order)
    for col, weight in weights.items():
        if col in input_weighted.columns:
            input_weighted[col] = input_weighted[col] * weight
            
    # --- STEP 4: FIND NEAREST NEIGHBOR ---
    # Find the single most similar user in the weighted space
    distances, indices = knn.kneighbors(input_weighted.values, n_neighbors=1)
    
    matched_index = indices[0][0]
    matched_user = db_profiles.iloc[matched_index]
    matched_user_id = matched_user['User_ID']
    
    # --- DEBUG REPORT ---
    print("\n" + "="*40)
    print(f"🔎 PLAN FOUND FOR: {user_input['Goal']} ({user_input['Frequency']} Days)")
    print("-" * 40)
    print(f"✅ MATCHED USER ID : {matched_user_id}")
    print(f"   • Goal Match    : {matched_user['Goal_x']}")
    print(f"   • Freq Match    : {matched_user['Workout_Frequency_x']} Days")
    print(f"   • Env Match     : {matched_user['Environment']}")
    print(f"   • Swim Ability  : {matched_user['Swim']} (User: {user_input['Swim']})")
    print("="*40)
    
    # --- STEP 5: RETRIEVE SCHEDULE ---
    schedule = db_schedule[db_schedule['User_ID'] == matched_user_id].copy()
    
    if 'Day' in schedule.columns:
        schedule = schedule.sort_values('Day')
        
    return schedule

## Model Usage Example

In [8]:
# EXAMPLE USAGE
if __name__ == "__main__":
    # Test User
    test_user = {
        'Age': 25, 'Gender': 'Male', 'Weight': 70,
        'Goal': 'Weight Loss', 'Frequency': 4, 
        'Level': 'Beginner', 'Environment': 'Home',
        'Badminton': 0, 'Football': 1, 'Basketball': 0, 
        'Volleyball': 0, 'Swim': 1 
    }

    # Run Prediction
    plan = predict_workout_plan(test_user)
    # Display Result
    if plan is not None:
        cols = ['Day', 'Muscle Group', 'Exercise Name', 'Sets', 'Reps', 'Calories_Burned']
        for day in plan['Day'].unique():
            print(f"\n📅 {day}")
            display(plan[plan['Day'] == day][cols].reset_index(drop=True))


🔎 PLAN FOUND FOR: Weight Loss (4 Days)
----------------------------------------
✅ MATCHED USER ID : 77
   • Goal Match    : Weight Loss
   • Freq Match    : 4 Days
   • Env Match     : Home
   • Swim Ability  : 1 (User: 1)

📅 Day 1 - Upper Strength


,Day,Muscle Group,Exercise Name,Sets,Reps,Calories_Burned
0,Day 1 - Upper Strength,Chest,decline push up,3,12-15,60
1,Day 1 - Upper Strength,Chest,diamond push up,3,12-15,60
2,Day 1 - Upper Strength,Back,superman,3,12-15,60
3,Day 1 - Upper Strength,Shoulders,inchworm,3,12-15,60
4,Day 1 - Upper Strength,Abs,commando plank,3,12-15,60



📅 Day 2 - Lower Quads


,Day,Muscle Group,Exercise Name,Sets,Reps,Calories_Burned
0,Day 2 - Lower Quads,Cardio,Bear Crawls,1,12 Mins,97
1,Day 2 - Lower Quads,Calves,standing calf raise,3,12-15,60
2,Day 2 - Lower Quads,Abs,side plank,3,12-15,60
3,Day 2 - Lower Quads,Quads,bulgarian split squat,3,12-15,60
4,Day 2 - Lower Quads,Quads,wall sit,3,12-15,60



📅 Day 3 - Upper Pump


,Day,Muscle Group,Exercise Name,Sets,Reps,Calories_Burned
0,Day 3 - Upper Pump,Biceps,superman,3,12-15,60
1,Day 3 - Upper Pump,Triceps,diamond push up,3,12-15,60
2,Day 3 - Upper Pump,Chest,decline push up,3,12-15,60
3,Day 3 - Upper Pump,Cardio,fast feet,1,12 Mins,97



📅 Day 4 - Lower Hams


,Day,Muscle Group,Exercise Name,Sets,Reps,Calories_Burned
0,Day 4 - Lower Hams,Calves,standing calf raise,3,12-15,60
1,Day 4 - Lower Hams,Hamstrings,inchworm,3,12-15,60
2,Day 4 - Lower Hams,Hamstrings,glute bridge,3,12-15,60
3,Day 4 - Lower Hams,Glutes,single leg glute bridge,3,12-15,60
4,Day 4 - Lower Hams,Cardio,Bear Crawls,1,10 Mins,81
